# Objective 1 — AI Trust Paradox: Data Cleaning Pipeline
**AICW Fellowship 2026 | Edunet Foundation | Galgotias University**  
**Faculty:** Dr. Chandni Rani | **Guide:** Ms. R Uma Maheswari

### ⚠️ This version runs on JupyterLite (jupyter.org/try-jupyter)
| Cell | What it does |
|------|--------------|
| 1 | Install libraries via micropip (Pyodide) |
| 2 | Upload `survey_results_public.csv` using file picker |
| 3–14 | All 13 cleaning steps (identical to Colab version) |
| 15 | Save + download `trust_paradox_clean.csv` |

> **Run cells one by one with Shift+Enter. Do NOT skip any cell.**


In [2]:
# CELL 1 — Install and import libraries (JupyterLite / Pyodide)

import sys

IN_PYODIDE = 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info')

if IN_PYODIDE:
    import micropip
    await micropip.install('pandas')
    await micropip.install('numpy')
    await micropip.install('scipy')
    print('Installed via micropip (Pyodide)')
else:
    import subprocess
    for pkg in ['pandas','numpy','scipy']:
        subprocess.call([sys.executable,'-m','pip','install',pkg,'-q'])
    print('Installed via pip (local Jupyter)')

import pandas as pd
import numpy  as np
import scipy                      # ← import the TOP-LEVEL scipy module
from scipy import stats
import warnings, io

warnings.filterwarnings('ignore')

print()
print('pandas  :', pd.__version__)
print('numpy   :', np.__version__)
print('scipy   :', scipy.__version__)   # ← scipy.__version__ NOT stats.__version__
print()
print('✓ All libraries ready')
print('  Run Cell 2 to upload your CSV file')


Installed via micropip (Pyodide)

pandas  : 2.3.2
numpy   : 2.2.5
scipy   : 1.14.1

✓ All libraries ready
  Run Cell 2 to upload your CSV file


In [3]:
# CELL 2 — Upload survey_results_public.csv

# Step 1: Install ipywidgets for JupyterLite
import micropip
await micropip.install('ipywidgets')

# Step 2: Import and show upload button
import ipywidgets as widgets
from IPython.display import display

upload_widget = widgets.FileUpload(
    accept      = '.csv',
    multiple    = False,
    description = 'Upload CSV',
    layout      = widgets.Layout(width='250px')
)

display(upload_widget)

print()
print('Click "Upload" above → select survey_results_public.csv')
print('File size is ~135 MB — wait until filename shows next to button.')
print('When upload is done → run Cell 3')

FileUpload(value=(), accept='.csv', description='Upload CSV', layout=Layout(width='250px'))


Click "Upload" above → select survey_results_public.csv
File size is ~135 MB — wait until filename shows next to button.
When upload is done → run Cell 3


In [4]:
# CELL 3 — STEP 1: Read uploaded file and verify

import io
import pandas as pd

# Check something was uploaded
if len(upload_widget.value) == 0:
    print('NO FILE UPLOADED YET')
    print('Go back to Cell 2, click Upload, select survey_results_public.csv')
    print('Then re-run this cell.')

else:
    # ── ipywidgets v8+ uses a TUPLE, not a dict ──
    uploaded_file = upload_widget.value[0]        # first item in tuple
    filename      = uploaded_file['name']          # filename string
    file_bytes    = bytes(uploaded_file['content']) # raw bytes

    size_mb = len(file_bytes) / 1024 / 1024
    print(f'Uploaded file : {filename}')
    print(f'Size          : {size_mb:.1f} MB')
    print()
    print('Loading CSV into pandas... (may take 30–60 seconds)')

    df = pd.read_csv(io.BytesIO(file_bytes), low_memory=False)

    print(f'Rows loaded   : {len(df):,}')
    print(f'Cols loaded   : {len(df.columns)}')

    # Hard verification
    assert len(df) == 49191,       f'Expected 49,191 rows — got {len(df):,}. Wrong file?'
    assert len(df.columns) == 172, f'Expected 172 columns — got {len(df.columns)}. Wrong file?'

    print()
    print('✓ Dataset verified: 49,191 rows × 172 columns')
    print('  Run Cell 4 — Duplicate check')

Uploaded file : survey_results_public.csv
Size          : 134.4 MB

Loading CSV into pandas... (may take 30–60 seconds)
Rows loaded   : 49,191
Cols loaded   : 172

✓ Dataset verified: 49,191 rows × 172 columns
  Run Cell 4 — Duplicate check


In [5]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — STEP 2: Duplicate check
# ════════════════════════════════════════════════════════════════

n_dupes = df.duplicated(subset=['ResponseId']).sum()
print(f'Duplicate rows found : {n_dupes}')

if n_dupes > 0:
    df = df.drop_duplicates(subset=['ResponseId'])
    print(f'Removed {n_dupes} duplicate rows.')
else:
    print('✓ Zero duplicates — dataset is clean at source')


Duplicate rows found : 0
✓ Zero duplicates — dataset is clean at source


In [6]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — STEP 3: Missing value profile (MNAR detection)
# ════════════════════════════════════════════════════════════════

KEY_COLS = [
    'AIAcc','AISent','AIThreat','AISelect','AIComplex',
    'AIFrustration','JobSat','YearsCode','DevType','Country',
    'Age','WorkExp','ToolCountWork',
]

print(f"  {'Column':<26} {'Non-Null':>9} {'Missing':>9} {'Missing %':>10}  Note")
print(f"  {'─'*72}")

for col in KEY_COLS:
    nn  = int(df[col].notna().sum())
    mis = len(df) - nn
    pct = mis / len(df) * 100
    note = '← MNAR — RETAIN, never delete' if pct > 30 else \
           '← cap outliers'               if col == 'YearsCode' else \
           '← complete ✓'                 if pct == 0 else ''
    print(f"  {col:<26} {nn:>9,} {mis:>9,} {pct:>9.1f}%  {note}")

print()
print('KEY INSIGHT: AIAcc (~32%), AISent (~32%), AISelect (~31%)')
print('are missing at almost the same rate — same ~15,500 rows.')
print('This is MNAR (Missing Not At Random).')
print('Decision: RETAIN all rows. Label blanks as "No Response".')


  Column                      Non-Null   Missing  Missing %  Note
  ────────────────────────────────────────────────────────────────────────
  AIAcc                         33,297    15,894      32.3%  ← MNAR — RETAIN, never delete
  AISent                        33,467    15,724      32.0%  ← MNAR — RETAIN, never delete
  AIThreat                      36,078    13,113      26.7%  
  AISelect                      33,720    15,471      31.5%  ← MNAR — RETAIN, never delete
  AIComplex                     33,283    15,908      32.3%  ← MNAR — RETAIN, never delete
  AIFrustration                 31,522    17,669      35.9%  ← MNAR — RETAIN, never delete
  JobSat                        26,670    22,521      45.8%  ← MNAR — RETAIN, never delete
  YearsCode                     43,042     6,149      12.5%  ← cap outliers
  DevType                       43,680     5,511      11.2%  
  Country                       35,437    13,754      28.0%  
  Age                           49,191         0   

In [7]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — STEP 4: Developer scope filter  (49,191 → 48,195)
# ════════════════════════════════════════════════════════════════

KEEP_MAINBRANCH = [
    'I am a developer by profession',
    'I am not primarily a developer, but I write code sometimes as part of my work/studies',
    'I am learning to code',
    'I code primarily as a hobby',
    'I used to be a developer by profession, but no longer am',
]

rows_before = len(df)
df = df[df['MainBranch'].isin(KEEP_MAINBRANCH)].copy()
rows_after  = len(df)
removed     = rows_before - rows_after

print(f'Rows before : {rows_before:,}')
print(f'Rows after  : {rows_after:,}')
print(f'Removed     : {removed}  (non-developer support roles)')

assert removed == 996,      f'Expected 996 removed, got {removed}'
assert rows_after == 48195, f'Expected 48,195 rows, got {rows_after:,}'

print()
for val, cnt in df['MainBranch'].value_counts().items():
    print(f'  {cnt:>7,}  {val[:70]}')

print(f'\n✓ Filter done: {rows_after:,} developer rows retained')


Rows before : 49,191
Rows after  : 48,195
Removed     : 996  (non-developer support roles)

   37,467  I am a developer by profession
    4,894  I am not primarily a developer, but I write code sometimes as part of 
    2,585  I am learning to code
    1,924  I code primarily as a hobby
    1,325  I used to be a developer by profession, but no longer am

✓ Filter done: 48,195 developer rows retained


In [8]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — STEP 5: Label blank values as 'No Response'
# Original columns untouched — new _clean columns created
# ════════════════════════════════════════════════════════════════

LABEL_COLS = {
    'AIAcc'    : 'AIAcc_clean',
    'AISent'   : 'AISent_clean',
    'AIThreat' : 'AIThreat_clean',
    'JobSat'   : 'JobSat_clean',
}

for src, dst in LABEL_COLS.items():
    df[dst] = df[src].fillna('No Response')
    nr_cnt  = (df[dst] == 'No Response').sum()
    blanks  = df[dst].isna().sum()
    print(f'  {src:<12} → {dst:<18}  {nr_cnt:>6,} labelled No Response  |  {blanks} blanks left')
    assert blanks == 0, f'ERROR: {dst} still has blank cells!'

print()
print('CORRECT formula: High Trust % = Highly trust count / answered count')
print('  = 1,000 / 32,717 = 3.1%  ✓')
print()
print('WRONG formula  : 1,000 / 48,195 = 2.1%  ✗  (wrong denominator)')
print()
print('✓ All _clean columns created with zero blanks')


  AIAcc        → AIAcc_clean         15,478 labelled No Response  |  0 blanks left
  AISent       → AISent_clean        15,311 labelled No Response  |  0 blanks left
  AIThreat     → AIThreat_clean      12,793 labelled No Response  |  0 blanks left
  JobSat       → JobSat_clean        21,525 labelled No Response  |  0 blanks left

CORRECT formula: High Trust % = Highly trust count / answered count
  = 1,000 / 32,717 = 3.1%  ✓

WRONG formula  : 1,000 / 48,195 = 2.1%  ✗  (wrong denominator)

✓ All _clean columns created with zero blanks


In [9]:
# ════════════════════════════════════════════════════════════════
# CELL 8 — STEP 6: Ordinal encoding (AIAcc_score and AISent_score)
# ════════════════════════════════════════════════════════════════

TRUST_MAP = {
    'Highly trust'              : 5,
    'Somewhat trust'            : 4,
    'Neither trust nor distrust': 3,
    'Somewhat distrust'         : 2,
    'Highly distrust'           : 1,
    'No Response'               : np.nan,
}

SENT_MAP = {
    'Very favorable'   : 5,
    'Favorable'        : 4,
    'Indifferent'      : 3,
    'Unfavorable'      : 2,
    'Very unfavorable' : 1,
    'Unsure'           : np.nan,
    'No Response'      : np.nan,
}

df['AIAcc_score']  = df['AIAcc_clean'].map(TRUST_MAP)
df['AISent_score'] = df['AISent'].map(SENT_MAP)

assert df.loc[df['AIAcc_clean']=='Highly trust',   'AIAcc_score'].iloc[0] == 5
assert df.loc[df['AIAcc_clean']=='Highly distrust','AIAcc_score'].iloc[0] == 1

mean_sc = df['AIAcc_score'].mean()
print(f'AIAcc_score mean     : {mean_sc:.3f}  (below 3.0 = leans toward distrust)')
print(f'AIAcc_score non-null : {df["AIAcc_score"].notna().sum():,}')
print()

trust_ans = df[df['AIAcc_clean'] != 'No Response']
print('Trust distribution (n=32,717 who answered):')
for level in ['Highly trust','Somewhat trust','Neither trust nor distrust',
               'Somewhat distrust','Highly distrust']:
    cnt = (trust_ans['AIAcc_clean'] == level).sum()
    pct = cnt / len(trust_ans) * 100
    bar = '█' * int(pct / 2)
    print(f'  {level:<35} {cnt:>6,}  {pct:>5.1f}%  {bar}')

print('\n✓ Ordinal encoding complete')


AIAcc_score mean     : 2.700  (below 3.0 = leans toward distrust)
AIAcc_score non-null : 32,717

Trust distribution (n=32,717 who answered):
  Highly trust                         1,000    3.1%  █
  Somewhat trust                       9,668   29.6%  ██████████████
  Neither trust nor distrust           7,039   21.5%  ██████████
  Somewhat distrust                    8,549   26.1%  █████████████
  Highly distrust                      6,461   19.7%  █████████

✓ Ordinal encoding complete


In [10]:
# ════════════════════════════════════════════════════════════════
# CELL 9 — STEP 7: YearsCode — cap outliers + ExperienceBand
# ════════════════════════════════════════════════════════════════

df['YearsCode_num'] = pd.to_numeric(df['YearsCode'], errors='coerce')

print(f'YearsCode max BEFORE cap : {df["YearsCode_num"].max():.0f}')
print(f'Values above 50          : {(df["YearsCode_num"] > 50).sum()}')

df['YearsCode_num'] = np.minimum(df['YearsCode_num'], 50)
print(f'YearsCode max AFTER cap  : {df["YearsCode_num"].max():.0f}  ✓')

def assign_band(years):
    if pd.isna(years):  return 'Unknown'
    elif years <= 2:    return 'Early Career (0-2 yrs)'
    elif years <= 5:    return 'Mid-Early (3-5 yrs)'
    elif years <= 10:   return 'Mid (6-10 yrs)'
    else:               return 'Experienced (10+ yrs)'

df['ExperienceBand'] = df['YearsCode_num'].apply(assign_band)

print('\nExperienceBand distribution:')
for band, cnt in df['ExperienceBand'].value_counts().items():
    pct = cnt / len(df) * 100
    print(f'  {band:<30} {cnt:>6,}  ({pct:.1f}%)')

assert df['ExperienceBand'].isna().sum() == 0
print('\n✓ YearsCode capped and ExperienceBand created')


YearsCode max BEFORE cap : 100
Values above 50          : 239
YearsCode max AFTER cap  : 50  ✓

ExperienceBand distribution:
  Experienced (10+ yrs)          25,523  (53.0%)
  Mid (6-10 yrs)                 10,192  (21.1%)
  Unknown                         5,905  (12.3%)
  Mid-Early (3-5 yrs)             4,993  (10.4%)
  Early Career (0-2 yrs)          1,582  (3.3%)

✓ YearsCode capped and ExperienceBand created


In [11]:
# ════════════════════════════════════════════════════════════════
# CELL 10 — STEP 8: Create UsesAI binary adoption flag
# ════════════════════════════════════════════════════════════════

USES_AI_MAP = {
    'Yes, I use AI tools daily'                   : 'Uses AI',
    'Yes, I use AI tools weekly'                  : 'Uses AI',
    'Yes, I use AI tools monthly or infrequently' : 'Uses AI',
    'No, but I plan to soon'                      : 'Does Not Use AI',
    "No, and I don't plan to"                     : 'Does Not Use AI',
}

def classify_ai(val):
    if pd.isna(val): return 'No Response'
    return USES_AI_MAP.get(val, 'Does Not Use AI')

df['UsesAI'] = df['AISelect'].apply(classify_ai)

print('UsesAI distribution:')
for cat, cnt in df['UsesAI'].value_counts().items():
    pct = cnt / len(df) * 100
    print(f'  {cat:<22} {cnt:>6,}  ({pct:.1f}%)')

answered   = df[df['UsesAI'] != 'No Response']
adopt_rate = (answered['UsesAI'] == 'Uses AI').mean() * 100
print(f'\nAdoption rate (among those who answered) : {adopt_rate:.1f}%')

assert abs(adopt_rate - 78.5) < 0.1, f'Unexpected adoption rate: {adopt_rate:.1f}%'
print('✓ UsesAI created — adoption rate confirmed at 78.5%')


UsesAI distribution:
  Uses AI                26,013  (54.0%)
  No Response            15,063  (31.3%)
  Does Not Use AI         7,119  (14.8%)

Adoption rate (among those who answered) : 78.5%
✓ UsesAI created — adoption rate confirmed at 78.5%


In [12]:
# ════════════════════════════════════════════════════════════════
# CELL 11 — STEP 9: ToolCountWork → ToolGroup
# ════════════════════════════════════════════════════════════════

tc = pd.to_numeric(df['ToolCountWork'], errors='coerce')

print(f'Non-null : {tc.notna().sum():,}')
print(f'Median   : {tc.median():.1f}  ← use this (not mean)')
print(f'Mean     : {tc.mean():.1f}  ← distorted by outliers')
print(f'Max      : {tc.max():.0f}  ← extreme outlier, ignore')

def assign_tool_group(val):
    if pd.isna(val): return 'No Response'
    elif val <= 1:   return '1 Tool'
    elif val <= 3:   return '2-3 Tools'
    else:            return '4+ Tools'

df['ToolGroup'] = tc.apply(assign_tool_group)

print('\nToolGroup distribution:')
for grp, cnt in df['ToolGroup'].value_counts().items():
    print(f'  {grp:<15} {cnt:>6,}')

print('\n✓ ToolGroup created')


Non-null : 27,119
Median   : 6.0  ← use this (not mean)
Mean     : 17.1  ← distorted by outliers
Max      : 10000  ← extreme outlier, ignore

ToolGroup distribution:
  4+ Tools        21,810
  No Response     21,076
  2-3 Tools        4,358
  1 Tool             951

✓ ToolGroup created


In [13]:
# ════════════════════════════════════════════════════════════════
# CELL 12 — STEP 10: Parse AIFrustration → 5 binary columns
# ════════════════════════════════════════════════════════════════

FRUSTRATION_KEYWORDS = {
    'Frust_AlmostRight'      : 'almost right',
    'Frust_Debugging'        : 'Debugging',
    'Frust_LessConfident'    : 'less confident',
    'Frust_HardToUnderstand' : 'hard to understand',
    'Frust_NoProblems'       : "haven't encountered",
}

print(f"  {'Column':<28} {'Count':>8}  Note")
print(f"  {'─'*50}")

for col_name, keyword in FRUSTRATION_KEYWORDS.items():
    mask         = df['AIFrustration'].str.contains(keyword, na=False, regex=False)
    count        = int(mask.sum())
    df[col_name] = mask.astype(int)
    print(f'  {col_name:<28} {count:>8,}  (1=has it, 0=does not)')

print('\n✓ Frustration binary columns created')


  Column                          Count  Note
  ──────────────────────────────────────────────────
  Frust_AlmostRight              20,465  (1=has it, 0=does not)
  Frust_Debugging                14,041  (1=has it, 0=does not)
  Frust_LessConfident             6,207  (1=has it, 0=does not)
  Frust_HardToUnderstand          5,050  (1=has it, 0=does not)
  Frust_NoProblems                    0  (1=has it, 0=does not)

✓ Frustration binary columns created


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 13 — STEP 11: Print all three paradox statistics
# ════════════════════════════════════════════════════════════════

trust_ans  = df[df['AIAcc_clean'] != 'No Response']
n_trust    = len(trust_ans)
adopt_rate = (df[df['UsesAI'] != 'No Response']['UsesAI'] == 'Uses AI').mean() * 100
ht_pct     = (trust_ans['AIAcc_clean'] == 'Highly trust').mean()  * 100
ad_pct     = trust_ans['AIAcc_clean'].isin(
                 ['Somewhat distrust','Highly distrust']).mean() * 100

BAND_ORDER = ['Early Career (0-2 yrs)','Mid-Early (3-5 yrs)',
               'Mid (6-10 yrs)','Experienced (10+ yrs)']

print('=' * 60)
print('  PARADOX 1 — USE-DESPITE-DISTRUST')
print('=' * 60)
print(f'  Adoption rate          : {adopt_rate:.1f}%')
print(f'  Highly trust %         : {ht_pct:.1f}%  (1 in 32 developers)')
print(f'  Any distrust %         : {ad_pct:.1f}%  (nearly half)')
print(f'  Paradox Severity Score : {adopt_rate - ht_pct:.1f} pts')
print(f'\n  Expertise effect (trust FALLS as experience RISES):')
for band in BAND_ORDER:
    g    = trust_ans[trust_ans['ExperienceBand'] == band]
    ht_g = (g['AIAcc_clean'] == 'Highly trust').mean() * 100
    ad_g = g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    print(f'    {band:<30}  High Trust {ht_g:>5.1f}%   Any Distrust {ad_g:>5.1f}%')

print('\n' + '=' * 60)
print('  PARADOX 2 — ADOPTION-ETHICS GAP')
print('=' * 60)
for grp in ['1 Tool','2-3 Tools','4+ Tools']:
    g  = df[(df['ToolGroup'] == grp) & (df['AIAcc_clean'] != 'No Response')]
    ad = g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    print(f'  {grp:<12}  Any Distrust: {ad:.1f}%   n={len(g):,}')

print('\n' + '=' * 60)
print('  PARADOX 3 — JOB THREAT (2025)')
print('=' * 60)
threat_ans = df[df['AIThreat_clean'] != 'No Response']
for val in ['No', "I'm not sure", 'Yes']:
    cnt   = (threat_ans['AIThreat_clean'] == val).sum()
    pct   = cnt / len(threat_ans) * 100
    label = {'No':'Not Threatened',"I'm not sure":'Uncertain','Yes':'Threatened'}[val]
    print(f'  {label:<20}  {cnt:>7,}  ({pct:.1f}%)')


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 14 — STEP 12: Final validation — ALL 9 checks must say PASS
# ════════════════════════════════════════════════════════════════

print('VALIDATION CHECKS\n')

checks = [
    ('Row count',                   len(df),                                            48195),
    ('Zero blanks: AIAcc_clean',    df['AIAcc_clean'].isna().sum(),                     0),
    ('Zero blanks: UsesAI',         df['UsesAI'].isna().sum(),                          0),
    ('Zero blanks: ExperienceBand', df['ExperienceBand'].isna().sum(),                  0),
    ('Zero blanks: AIThreat_clean', df['AIThreat_clean'].isna().sum(),                  0),
    ('Highly trust count',          int((df['AIAcc_clean']=='Highly trust').sum()),      1000),
    ('Uses AI count',               int((df['UsesAI']=='Uses AI').sum()),                26013),
    ('Experienced band count',      int((df['ExperienceBand']=='Experienced (10+ yrs)').sum()), 25523),
    ('No Response AIAcc count',     int((df['AIAcc_clean']=='No Response').sum()),       15478),
]

all_pass = True
for label, actual, expected in checks:
    ok     = (actual == expected)
    status = 'PASS ✓' if ok else f'FAIL ✗  (expected {expected:,})'
    print(f'  {label:<38}  {str(actual):>8}  {status}')
    if not ok:
        all_pass = False

print()
if all_pass:
    print('✓ ALL 9 CHECKS PASSED — dataset is clean and verified')
    print('  Run Cell 15 to save and download the cleaned CSV')
else:
    print('✗ SOME CHECKS FAILED — review output above before saving')


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 15 — STEP 13: Save cleaned CSV + trigger browser download
#
# JupyterLite cannot write to your local disk directly.
# This cell saves the file to the browser's virtual filesystem
# AND triggers a download so it saves to your Downloads folder.
# ════════════════════════════════════════════════════════════════

OUTPUT_FILENAME = 'trust_paradox_clean.csv'

# Step 1 — Convert dataframe to CSV bytes
csv_bytes = df.to_csv(index=False).encode('utf-8')

print(f'Rows  : {len(df):,}')
print(f'Cols  : {len(df.columns)}  ({len(df.columns) - 172} new columns added)')
print(f'Size  : {len(csv_bytes)/1024/1024:.1f} MB')
print()

# Step 2 — Write to JupyterLite virtual filesystem
with open(OUTPUT_FILENAME, 'wb') as f:
    f.write(csv_bytes)
print(f'Saved to virtual filesystem: {OUTPUT_FILENAME}')

# Step 3 — Trigger browser download using JavaScript
import base64
from IPython.display import Javascript, display

b64 = base64.b64encode(csv_bytes).decode('utf-8')

js_code = f"""
var bytes   = Uint8Array.from(atob('{b64}'), c => c.charCodeAt(0));
var blob    = new Blob([bytes], {{type: 'text/csv'}});
var url     = URL.createObjectURL(blob);
var a       = document.createElement('a');
a.href      = url;
a.download  = '{OUTPUT_FILENAME}';
document.body.appendChild(a);
a.click();
document.body.removeChild(a);
URL.revokeObjectURL(url);
console.log('Download triggered: {OUTPUT_FILENAME}');
"""

display(Javascript(js_code))

print()
print('Browser download triggered — check your Downloads folder.')
print()

new_cols = [
    'YearsCode_num','ExperienceBand',
    'AIAcc_clean','AIAcc_score',
    'AISent_clean','AISent_score',
    'AIThreat_clean','JobSat_clean',
    'UsesAI','ToolGroup',
    'Frust_AlmostRight','Frust_Debugging',
    'Frust_LessConfident','Frust_HardToUnderstand','Frust_NoProblems',
]
print('New columns added:')
for c in new_cols:
    print(f'  {c}')

print()
print('OBJECTIVE 1 COMPLETE')
print('Next: run Objective_2 notebook with trust_paradox_clean.csv as input')
